# L14 · Árboles y bosques causales

**S14 · Lectura: Cap. 13 · 40–55 min**

## La pregunta

¿La heterogeneidad es señal o ruido?

## Qué harás

Entender honestidad, partición y estabilidad en bosques causales.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Estimaciones por grupo y pruebas de estabilidad.

> **Idea clave:** Separar dónde se busca una división de dónde se estima el efecto reduce el sobreajuste.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
rng = setup()
N_REP = R(30, 300)
B_ARB = R(40, 200)
print("réplicas:", N_REP, "· árboles del bosque:", B_ARB)

## 1. Honestidad (Def. 13.1, §13.1)
Un árbol que **elige** los cortes con unos datos y **estima** el efecto de cada hoja con los mismos datos se engaña: los cortes se eligieron justo donde el ruido favorecía una diferencia. La receta honesta:

1. Partir la muestra en $S_{\text{split}}$ y $S_{\text{est}}$.
2. Construir el árbol (los cortes) con $S_{\text{split}}$.
3. Estimar el efecto de cada hoja con $S_{\text{est}}$, con su error estándar.

Para la variable que el árbol predice usamos el **score doblemente robusto** (AIPW) con nuisance ajustados por *cross-fitting* en dos mitades, de modo que la media de ese score dentro de una hoja estima el efecto de la hoja aun con confusión observada.

In [ ]:
def make_dgp(n=2000, p=10, seed=0, het=1.0):
    r = np.random.default_rng(seed); X = r.uniform(-1, 1, (n, p)); e = 1 / (1 + np.exp(-1.0 - 0.5 * X[:, 0]))
    D = (r.uniform(size=n) < e).astype(int); tau = 2 + het * (3 * X[:, 0] - X[:, 1] ** 2); mu0 = X[:, 2] + .5 * X[:, 3]
    Y = mu0 + D * tau + r.standard_normal(n) * .5
    return Y, D, X, tau

ETR = lambda: ExtraTreesRegressor(n_estimators=40, min_samples_leaf=5, max_features=0.8, random_state=0)
def psi_cf(X, y, d, K=2, seed=SEED):
    """Score AIPW por unidad con nuisance de otros pliegues (cross-fitting)."""
    n = len(y); psi = np.zeros(n)
    for te in kfold(n, K, seed):
        tr = np.setdiff1d(np.arange(n), te); t1, t0 = tr[d[tr] == 1], tr[d[tr] == 0]
        m1 = ETR().fit(X[t1], y[t1]).predict(X[te]); m0 = ETR().fit(X[t0], y[t0]).predict(X[te])
        e = np.clip(LogisticRegression(max_iter=500).fit(X[tr], d[tr]).predict_proba(X[te])[:, 1], .02, .98)
        psi[te] = m1 - m0 + d[te] * (y[te] - m1) / e - (1 - d[te]) * (y[te] - m0) / (1 - e)
    return psi

Y, D, X, tau = make_dgp(2000, seed=42)
psi = psi_cf(X, Y, D)
print("media del score AIPW:", psi.mean().round(3), "· τ medio verdadero:", tau.mean().round(3))

### Completa tú: la estimación honesta de las hojas
`hojas_honestas(arbol, X_est, psi_est)` recibe un árbol ya construido (sklearn) y la muestra de **estimación**. Debe devolver, **por hoja**, `n`, `tau` (media de `psi_est` en la hoja), `se` ($=\mathrm{sd}/\sqrt n$) y el IC 95 %. Pista: `arbol.apply(X)` da el índice de hoja de cada fila.

In [ ]:
# ✏️ Completa tú — hojas_honestas
# Pista: Para cada hoja h: v = psi_est[hoja == h]; tau = v.mean(); se = v.std(ddof=1)/sqrt(len(v)).
def hojas_honestas(arbol, X_est, psi_est):
    """DataFrame indexado por hoja con columnas n, tau, se, lo, hi (usando SOLO la muestra de estimación)."""
    hoja = arbol.apply(X_est)
    raise NotImplementedError("completa hojas_honestas")

In [ ]:
n = len(Y); idx = np.random.default_rng(0).permutation(n); A, Bm = idx[: n // 2], idx[n // 2:]
arb = DecisionTreeRegressor(min_samples_leaf=50, max_leaf_nodes=6, random_state=0).fit(X[A], psi[A])      # cortes con S_split
ho = hojas_honestas(arb, X[Bm], psi[Bm])                                                                    # efectos con S_est
lab_check("hojas_honestas (estructura)", lambda: (list(ho.columns) == ["n", "tau", "se", "lo", "hi"], int(ho["n"].sum()) == len(Bm),
          bool(np.all(ho["lo"] < ho["tau"])), bool(np.allclose(ho.loc[ho.index[0], "se"], psi[Bm][arb.apply(X[Bm]) == ho.index[0]].std(ddof=1) / np.sqrt(ho.loc[ho.index[0], "n"])))),
          [((), (True, True, True, True), 0)])
print(export_text(arb, feature_names=[f"X{i}" for i in range(X.shape[1])], max_depth=3))
# EVALUATION-CELL
ho["tau_verdadero_medio"] = [tau[Bm][arb.apply(X[Bm]) == h].mean() for h in ho.index]
show(ho.round(3))

**Mira el resultado.** ¿Qué variables eligió el árbol para partir, y coincide con la fórmula $\tau=2+3X_0-X_1^2$?

**Respuesta:**

Debería partir sobre todo por $X_0$ (efecto lineal fuerte) y, en menor medida, por $X_1$. Si aparecen cortes sobre otras variables, son ruido: por eso se usa `min_samples_leaf` y por eso la honestidad importa: ese ruido **no** contamina los efectos de las hojas, porque se estiman en otra muestra.



## 2. Por qué la honestidad restaura la cobertura
Repetimos el proceso muchas veces y medimos con qué frecuencia el IC 95 % de una hoja contiene el efecto verdadero medio de esa hoja:
- **Honesto:** cortes con $S_{\text{split}}$, efectos con $S_{\text{est}}$.
- **No honesto:** cortes **y** efectos con la misma muestra.

In [ ]:
def cobertura(honesto, reps=N_REP, n=1000):
    cub = []
    for s in range(reps):
        Y_, D_, X_, tau_ = make_dgp(n, seed=500 + s); ps = psi_cf(X_, Y_, D_)
        idx_ = np.random.default_rng(s).permutation(n); A_, B_ = idx_[: n // 2], idx_[n // 2:]
        arb_ = DecisionTreeRegressor(min_samples_leaf=25, max_leaf_nodes=6, random_state=0).fit(X_[A_], ps[A_])
        E = B_ if honesto else A_                                       # muestra con la que se estima
        h_ = hojas_honestas(arb_, X_[E], ps[E])
        for h in h_.index:
            real = tau_[E][arb_.apply(X_[E]) == h].mean(); cub.append(h_.loc[h, "lo"] <= real <= h_.loc[h, "hi"])
    return float(np.mean(cub))
cob = pd.Series({"honesto (estima en otra muestra)": cobertura(True), "no honesto (misma muestra)": cobertura(False)}, name="cobertura del IC 95 % por hoja")
show(cob.round(3).to_frame())

**Mira el resultado.** ¿Cuál de las dos coberturas está más cerca de 0,95 y por qué la otra se queda corta?

**Respuesta:**

La honesta: la hoja ya está fijada cuando se estima, así que el IC por hoja se comporta como un IC ordinario. En la no honesta, los cortes se eligieron *porque* separaban efectos grandes de pequeños en esa misma muestra; las medias de hoja quedan sesgadas hacia los extremos y los IC, demasiado optimistas. (La cobertura honesta puede quedar algo por debajo de 0,95 por el error en los nuisance y por hojas pequeñas.)



## 3. Señal de alerta: buscar subgrupos «a ojo»
El libro advierte contra recorrer subgrupos hasta encontrar uno con efecto. Simulamos un mundo **sin ningún efecto** ($\tau=0$) y comparamos: (a) dividir por cada una de 10 covariables en la mediana (20 subgrupos) y *reportar el más llamativo*; (b) reportar una hoja de un árbol honesto elegida **antes** de mirar (la de mayor tamaño).

In [ ]:
def nulo(reps=max(100, N_REP * 4), n=600, p=10):
    r = np.random.default_rng(77); a_ojo, honesta = [], []
    for _ in range(reps):
        X_ = r.normal(size=(n, p)); D_ = r.binomial(1, .5, n); Y_ = X_[:, 0] + r.normal(size=n)          # τ = 0
        ys = Y_ * (D_ - .5) / .25                                                                       # resultado transformado (e = 0,5 conocido)
        mejor = 0.0
        for j in range(p):
            for lado in (X_[:, j] < np.median(X_[:, j]), X_[:, j] >= np.median(X_[:, j])):
                v = ys[lado]; mejor = max(mejor, abs(v.mean() / (v.std(ddof=1) / np.sqrt(len(v)))))
        a_ojo.append(mejor < 1.96)                              # True = el IC del subgrupo «más llamativo» CONTIENE 0 (cobertura correcta)
        idx_ = r.permutation(n); A_, B_ = idx_[: n // 2], idx_[n // 2:]
        ar = DecisionTreeRegressor(min_samples_leaf=40, max_leaf_nodes=5, random_state=0).fit(X_[A_], ys[A_])
        h_ = hojas_honestas(ar, X_[B_], ys[B_]); g = h_["n"].idxmax()
        honesta.append(h_.loc[g, "lo"] <= 0 <= h_.loc[g, "hi"])
    return np.mean(a_ojo), np.mean(honesta)
a, h = nulo()
show(pd.Series({"subgrupo más llamativo entre 20 (a ojo)": a, "hoja honesta elegida de antemano": h}, name="el IC contiene el efecto verdadero (0)").round(3).to_frame())

**Mira el resultado.** El efecto verdadero es 0 en todos los subgrupos. ¿Por qué el subgrupo «más llamativo» deja de contener el 0 con tanta frecuencia?

**Respuesta:**

Porque al elegir el **máximo** de 20 estadísticos se garantiza (con probabilidad alta) que alguno supere 1,96 por azar: con 20 subgrupos independientes, la probabilidad de que todos queden dentro sería $0{,}95^{20}\approx0{,}36$. Aquí la cobertura es algo mayor porque los subgrupos están correlacionados (mitades complementarias de la misma variable), pero sigue muy por debajo de 0,95: el procedimiento no tiene el nivel que dice tener. La hoja honesta elegida antes de mirar sí lo tiene (≈ 0,95).



## 4. Bosque honesto (§13.3)
Un árbol es inestable; un **bosque** promedia muchos árboles, cada uno construido con una submuestra distinta y honesto internamente. Implementación mínima: para cada árbol, tomar el 50 % de las filas, partirlas en $S_{\text{split}}$/$S_{\text{est}}$, construir el árbol y calcular los efectos de sus hojas; la predicción es el promedio sobre árboles.

In [ ]:
def bosque_honesto(X, psi, Xnew, B=B_ARB, min_leaf=20, seed=0, max_feat=0.5):
    r = np.random.default_rng(seed); n = len(psi); pred = np.zeros((B, len(Xnew))); imp = np.zeros(X.shape[1])
    for b in range(B):
        sub = r.choice(n, n // 2, replace=False); A_, E_ = sub[: len(sub) // 2], sub[len(sub) // 2:]
        ar = DecisionTreeRegressor(min_samples_leaf=min_leaf, max_features=max_feat, random_state=int(r.integers(1e9))).fit(X[A_], psi[A_])
        hoja_e = ar.apply(X[E_]); efecto = {h: psi[E_][hoja_e == h].mean() for h in np.unique(hoja_e)}
        fallback = psi[E_].mean(); hn = ar.apply(Xnew)
        pred[b] = [efecto.get(h, fallback) for h in hn]; imp += ar.feature_importances_
    return pred.mean(0), imp / B
Xt, tt = make_dgp(2000, seed=7)[2:]
tau_f, imp_f = bosque_honesto(X, psi, Xt)
# EVALUATION-CELL
print("RMSE del bosque frente al τ verdadero:", round(float(np.sqrt(np.mean((tau_f - tt) ** 2))), 3), "· correlación:", round(float(np.corrcoef(tau_f, tt)[0, 1]), 3))
show(pd.Series(imp_f, index=[f"X{i}" for i in range(10)], name="importancia (reducción de impureza media)").sort_values(ascending=False).head(5).round(3).to_frame())

**Mira el resultado.** La importancia de variables del bosque, ¿dice «qué variables *causan* el efecto» o «qué variables *predicen* el efecto»?

**Respuesta:**

Dice cuáles se usaron para **partir** el espacio de efectos, es decir, cuáles *se asocian* con la heterogeneidad estimada. No es causal: dos covariables correlacionadas se reparten la importancia, y una variable irrelevante puede aparecer si es proxy de una relevante. Es una pista para investigar, no un hallazgo.



## 5. `min_samples_leaf`: bias-varianza de la partición (ejercicio)
Hojas más pequeñas = más flexibilidad pero estimaciones más ruidosas. Probamos `min_samples_leaf ∈ {5, 10, 20, 50}` en el bosque.

In [ ]:
filas = []
for ml_ in (5, 10, 20, 50):
    th, _ = bosque_honesto(X, psi, Xt, min_leaf=ml_)
    filas.append((ml_, float(np.sqrt(np.mean((th - tt) ** 2))), float(np.mean(th - tt))))
show(pd.DataFrame(filas, columns=["min_samples_leaf", "RMSE", "sesgo medio"]).set_index("min_samples_leaf").round(3))   # EVALUATION-CELL

## 6. Contra el X-learner (nombre correcto, E11)
El Listado 13.2 del libro se titula «vs. R-learner» pero usa `XLearner`: aquí se llama por su nombre. Comparamos el bosque honesto con el **X-learner** (boosting) sobre `real/ihdp`, que trae el efecto individual.

In [ ]:
sc = load_scenario("real/ihdp"); sc.card()
X1, Y1, D1 = sc.xyt("train"); D1 = D1.astype(int); X2, _, _ = sc.xyt("test")
def x_learner(Y, D, X, Xnew):
    mk = lambda: HistGradientBoostingRegressor(max_iter=100, learning_rate=.1, max_depth=3, random_state=0)
    m1, m0 = mk().fit(X[D == 1], Y[D == 1]), mk().fit(X[D == 0], Y[D == 0])
    g1 = mk().fit(X[D == 1], Y[D == 1] - m0.predict(X[D == 1])); g0 = mk().fit(X[D == 0], m1.predict(X[D == 0]) - Y[D == 0])
    e = LogisticRegression(max_iter=500).fit(X, D).predict_proba(Xnew)[:, 1]
    return e * g0.predict(Xnew) + (1 - e) * g1.predict(Xnew)
psi1 = psi_cf(X1, Y1, D1)
tau_bosque, imp_ihdp = bosque_honesto(X1, psi1, X2, min_leaf=10)
tau_xl = x_learner(Y1, D1, X1, X2)
# EVALUATION-CELL
tv = sc.tau("test")
if tv is not None:
    show(pd.DataFrame({"PEHE": {"bosque honesto": np.sqrt(np.mean((tau_bosque - tv) ** 2)), "X-learner": np.sqrt(np.mean((tau_xl - tv) ** 2))},
                       "corr. con τ": {"bosque honesto": np.corrcoef(tau_bosque, tv)[0, 1], "X-learner": np.corrcoef(tau_xl, tv)[0, 1]}}).round(3))
    print("filas de prueba:", len(tv))

In [ ]:
# Vista rápida de los datos del escenario
import matplotlib.pyplot as plt
import numpy as np
_fig, _ax = plt.subplots(1, 2, figsize=(9, 3.2))
try:
    _scenario = globals().get("sc") or globals().get("lc")
    if _scenario is None:
        raise NameError("No hay un escenario cargado para el gráfico editorial")
    _df = _scenario.train
    if _scenario.treatment is not None and _scenario.outcome is not None:
        _t = _df[_scenario.treatment].to_numpy(float)
        _y = _df[_scenario.outcome].to_numpy(float)
        if np.unique(_t).size == 2:
            _counts = [int((_t == value).sum()) for value in (0, 1)]
            _ax[0].bar(["control", "tratado"], _counts)
            _ax[0].set_title("Asignación")
            _ax[1].boxplot([_y[_t == 0], _y[_t == 1]], labels=["control", "tratado"])
            _ax[1].set_title("Resultado observado")
        else:
            _ax[0].hist(_t, bins=20)
            _ax[0].set_title(_scenario.treatment)
            _ax[1].scatter(_t, _y, s=8, alpha=.35)
            _ax[1].set_title(f"{_scenario.treatment} frente a {_scenario.outcome}")
    elif _scenario.adj is not None:
        _ax[0].imshow(_scenario.adj, cmap="Greys", interpolation="nearest")
        _ax[0].set_title("Grafo del escenario")
        _ax[1].axis("off")
    else:
        _ax[0].axis("off"); _ax[1].axis("off")
    _fig.suptitle(f"Escenario: {_scenario.name}")
    _fig.tight_layout(); plt.show()
finally:
    plt.close(_fig)

**Mira el resultado.** Si el bosque y el X-learner tienen una correlación parecida con el τ verdadero pero PEHE distinto, ¿qué te dice?

**Respuesta:**

Que ordenan a las unidades de forma parecida pero **no ponen la misma escala**: uno puede estar encogiendo o inflando los efectos. Para focalizar un programa (elegir al 30 % con más efecto) basta el orden; para informar "cuánto" se beneficia alguien, el PEHE y el sesgo importan. Con ~450 filas de entrenamiento y 150 de prueba, además, una diferencia pequeña puede ser ruido.



## 7. Estabilidad entre semillas
Un bosque depende de la semilla (submuestras y cortes). Si dos semillas dan CATE muy distintos, la heterogeneidad que ves **no es una propiedad de los datos**. Entrenamos con 5 semillas y medimos acuerdo.

In [ ]:
preds = np.array([bosque_honesto(X1, psi1, X2, min_leaf=10, seed=s)[0] for s in range(5)])
corr = np.corrcoef(preds); off = corr[np.triu_indices(5, 1)]
print(f"correlación entre CATE de semillas distintas: media {off.mean():.3f} · mínima {off.min():.3f}")
print(f"desviación estándar del CATE de una fila entre semillas (media sobre filas): {preds.std(0).mean():.3f}  · desviación del CATE entre filas: {preds.mean(0).std():.3f}")
top = [set(int(j) for j in np.argsort(-bosque_honesto(X1, psi1, X2, min_leaf=10, seed=s)[1])[:3]) for s in range(5)]
print("variables del top-3 de importancia por semilla:", [sorted(t) for t in top])
estable = off.mean() > .8

**Mira el resultado.** Si la correlación entre semillas fuera baja (< 0,5), ¿qué harías antes de usar los CATE para decidir?

**Respuesta:**

No usarlos aún: más árboles, `min_samples_leaf` mayor o un modelo más simple. Una heterogeneidad que cambia con la semilla es varianza de Monte Carlo del algoritmo, no señal. Con pocas filas de entrenamiento (IHDP real: ~450) es habitual que la estabilidad sea moderada; por eso se reporta.



## 8. Anticipo: de CATE a política (Cap. 21)
Un **árbol de política** resume los CATE en reglas simples tratar/no tratar. Ajustamos un árbol de profundidad 2 sobre el CATE estimado y leemos las reglas (sin evaluarlas: eso es L22).

In [ ]:
cov = list(sc.covariates)
pol = DecisionTreeRegressor(max_depth=2, min_samples_leaf=20, random_state=0).fit(X1, bosque_honesto(X1, psi1, X1, min_leaf=10)[0])
print(export_text(pol, feature_names=cov))

## 9. Supuesto y bitácora

In [ ]:
fila_b = bitacora(
    pregunta="¿Qué heterogeneidad del efecto hay en IHDP y es estable e independiente de la semilla?",
    metodo="Bosque causal honesto (AIPW + cross-fitting, 50 % de submuestra por árbol) frente a X-learner",
    estimando="CATE τ(x) en IHDP",
    numero_ic=f"acuerdo entre 5 semillas: correlación media {off.mean():.2f} (mín. {off.min():.2f})",
    supuesto=supuesto("observable", "medido", f"estabilidad entre 5 semillas del bosque: {'aceptable (> 0,8)' if estable else 'insuficiente (≤ 0,8)'}"),
    decision="continuar" if estable else "alertar", lab="L14")

---
## Qué queda para el libro
Quedan en el libro: `CausalForestDML` y GRF con sus intervalos (infinitesimal jackknife), el criterio de partición exacto de la Alg. 6 y los árboles de política con valor esperado (se anticipan en el §8). El caso NSW no está en el benchmark.

*Material formativo (sin nota). Los datos son sintéticos: ver la ficha del escenario.*